# Violent Crime Capstone

**Main question:** How well can socioeconomic factors predict violent crime rates?  
**Subquestion:** Which categories are the strongest predictors of violent crime?

This notebook cleans and merges:
- FBI violent crime data (2024)
- BLS county unemployment data (2024)
- ACS 5-year county income, poverty, and education data (2024 release)


## 1. Imports and setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

BASE_DIR = Path.cwd()
CLEAN_DATA = BASE_DIR / 'cleaned_data'
OUTPUTS = BASE_DIR / 'outputs'

CLEAN_DATA.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. Load and clean FBI violent crime data

In [2]:
crime_file = BASE_DIR / 'CIUS_Table_10_Offenses_Known_to_Law_Enforcement_by_State_by_Metropolitan_and_Nonmetropolitan_Counties_2024.xlsx'

crime = pd.read_excel(crime_file, header=4)

crime.columns = (
    crime.columns.astype(str)
    .str.strip()
    .str.replace('\\n', '_', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
    .str.lower()
)

crime = crime.rename(columns={
    'larceny__theft': 'larceny_theft',
    'arson1': 'arson'
})

crime.head()

,state,metropolitan_nonmetropolitan,county,violentcrime,murder_andnonnegligentmanslaughter,rape,robbery,aggravatedassault,propertycrime,burglary,larceny_theft,motorvehicletheft,arson
0,ALABAMA,Metropolitan County,Autauga,54.000,1.000,7.000,0.000,46.000,221.000,46.000,143.000,32.000,1.000
1,ALABAMA,Metropolitan County,Baldwin,137.000,0.000,4.000,0.000,133.000,170.000,23.000,142.000,5.000,2.000
2,ALABAMA,Metropolitan County,Bibb,34.000,0.000,5.000,0.000,29.000,78.000,26.000,43.000,9.000,0.000
3,ALABAMA,Metropolitan County,Blount,82.000,1.000,18.000,2.000,61.000,298.000,66.000,180.000,52.000,2.000
4,ALABAMA,Metropolitan County,Calhoun,238.000,2.000,5.000,0.000,231.000,192.000,48.000,144.000,0.000,8.000


In [3]:
crime = crime.dropna(subset=['county', 'violent_crime']).copy()
crime['arson'] = crime['arson'].fillna(0)

crime_cols = [
    'violent_crime',
    'murder_and_nonnegligent_manslaughter',
    'rape',
    'robbery',
    'aggravated_assault',
    'property_crime',
    'burglary',
    'larceny_theft',
    'motor_vehicle_theft',
    'arson'
]

crime[crime_cols] = crime[crime_cols].apply(pd.to_numeric, errors='coerce')
crime[crime_cols] = crime[crime_cols].astype(int)
crime['year'] = 2024

crime.info()

KeyError: ['violent_crime']

In [ ]:
state_to_abbr = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE',
    'FLORIDA': 'FL', 'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID',
    'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS',
    'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS',
    'MISSOURI': 'MO', 'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV',
    'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH', 'OKLAHOMA': 'OK',
    'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT',
    'VERMONT': 'VT', 'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY', 'DISTRICT OF COLUMBIA': 'DC'
}


def clean_county_name(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(' COUNTY', '', regex=False)
        .str.replace(' PARISH', '', regex=False)
        .str.replace(' BOROUGH', '', regex=False)
        .str.replace(' CENSUS AREA', '', regex=False)
        .str.replace(' MUNICIPALITY', '', regex=False)
        .str.replace(' CITY AND BOROUGH OF ', '', regex=False)
        .str.replace(' CITY OF ', '', regex=False)
        .str.replace(' ST.', 'SAINT', regex=False)
        .str.replace(' ST ', 'SAINT ', regex=False)
        .str.replace('.', '', regex=False)
    )

crime['state'] = (
    crime['state']
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r'[^A-Z ]', '', regex=True)
    .str.strip()
)

crime = crime[
    ~crime['county'].astype(str).str.contains(
        'Police Department|Sheriff|Department of Public Safety|Marshal|Township|Village|City of|Public Safety',
        case=False,
        na=False
    )
].copy()

crime['state_abbr'] = crime['state'].map(state_to_abbr)
crime['county_clean'] = clean_county_name(crime['county'])

crime['county_clean'] = crime['county_clean'].replace({
    'DEWITT': 'DE WITT',
    'LA SALLE': 'LASALLE',
    'ANACONDA-DEER LODGE': 'ANACONDA DEER LODGE',
    'BUTTE-SILVER BOW': 'BUTTE SILVER BOW',
    'HARTSVILLE/TROUSDALE': 'TROUSDALE'
})

crime[['state', 'state_abbr', 'county', 'county_clean']].head()

In [ ]:
crime.to_csv(CLEAN_DATA / 'crime_cleaned.csv', index=False)

## 3. Load and clean BLS county unemployment data

In [ ]:
bls_file = BASE_DIR / 'laucnty24.xlsx'
bls = pd.read_excel(bls_file, header=1)

bls.columns = (
    bls.columns.astype(str)
    .str.strip()
    .str.replace('\\n', '_', regex=False)
    .str.replace(' ', '_', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('%', 'pct', regex=False)
    .str.replace(r'[^A-Za-z0-9_]', '', regex=True)
    .str.lower()
)

bls = bls.rename(columns={
    'county_name_state_abbreviation': 'county_name',
    'unemployment_rate_pct': 'unemployment_rate'
})

bls = bls.dropna(subset=['state_fips_code', 'county_fips_code']).copy()
bls['state_fips_code'] = bls['state_fips_code'].astype(int).astype(str).str.zfill(2)
bls['county_fips_code'] = bls['county_fips_code'].astype(int).astype(str).str.zfill(3)
bls['fips'] = bls['state_fips_code'] + bls['county_fips_code']

num_cols = ['year', 'labor_force', 'employed', 'unemployed', 'unemployment_rate']
bls[num_cols] = bls[num_cols].apply(pd.to_numeric, errors='coerce')

bls[['county_clean', 'state_abbr']] = bls['county_name'].str.split(',', n=1, expand=True)
bls['county_clean'] = clean_county_name(bls['county_clean'])
bls['state_abbr'] = bls['state_abbr'].str.strip()

bls = bls[[
    'fips', 'county_name', 'state_abbr', 'county_clean', 'year',
    'labor_force', 'employed', 'unemployed', 'unemployment_rate'
]]

bls.head()

## 4. Merge FBI and BLS data

In [ ]:
merged = crime.merge(
    bls,
    on=['state_abbr', 'county_clean', 'year'],
    how='left',
    suffixes=('_crime', '_bls')
)

print('Missing unemployment_rate values:', merged['unemployment_rate'].isna().sum())
merged[merged['unemployment_rate'].isna()][['state', 'county']].drop_duplicates().head(20)

In [ ]:
merged.to_csv(CLEAN_DATA / 'crime_bls_merged.csv', index=False)

## 5. Pull ACS income data and merge

In [ ]:
acs_income_url = 'https://api.census.gov/data/2024/acs/acs5?get=NAME,B19013_001E&for=county:*'

acs_income = pd.read_json(acs_income_url)
acs_income.columns = acs_income.iloc[0]
acs_income = acs_income[1:].copy()

acs_income = acs_income.rename(columns={
    'NAME': 'county_name',
    'B19013_001E': 'median_household_income',
    'state': 'state_fips',
    'county': 'county_fips'
})

acs_income['state_fips'] = acs_income['state_fips'].astype(str).str.zfill(2)
acs_income['county_fips'] = acs_income['county_fips'].astype(str).str.zfill(3)
acs_income['fips'] = acs_income['state_fips'] + acs_income['county_fips']
acs_income['median_household_income'] = pd.to_numeric(acs_income['median_household_income'], errors='coerce')

acs_income = acs_income[['fips', 'county_name', 'median_household_income']]
acs_income.head()

In [ ]:
merged = merged.drop(columns=['median_household_income'], errors='ignore')

merged = merged.merge(
    acs_income[['fips', 'median_household_income']],
    on='fips',
    how='left'
)

print('Missing median household income values:', merged['median_household_income'].isna().sum())

## 6. Pull ACS poverty data and merge

In [ ]:
acs_poverty_url = 'https://api.census.gov/data/2024/acs/acs5/subject?get=NAME,S1701_C03_001E&for=county:*'

acs_poverty = pd.read_json(acs_poverty_url)
acs_poverty.columns = acs_poverty.iloc[0]
acs_poverty = acs_poverty[1:].copy()

acs_poverty = acs_poverty.rename(columns={
    'NAME': 'county_name',
    'S1701_C03_001E': 'poverty_rate',
    'state': 'state_fips',
    'county': 'county_fips'
})

acs_poverty['state_fips'] = acs_poverty['state_fips'].astype(str).str.zfill(2)
acs_poverty['county_fips'] = acs_poverty['county_fips'].astype(str).str.zfill(3)
acs_poverty['fips'] = acs_poverty['state_fips'] + acs_poverty['county_fips']
acs_poverty['poverty_rate'] = pd.to_numeric(acs_poverty['poverty_rate'], errors='coerce')

acs_poverty = acs_poverty[['fips', 'county_name', 'poverty_rate']]
acs_poverty.head()

In [ ]:
merged = merged.drop(columns=['poverty_rate'], errors='ignore')

merged = merged.merge(
    acs_poverty[['fips', 'poverty_rate']],
    on='fips',
    how='left'
)

print('Missing poverty_rate values:', merged['poverty_rate'].isna().sum())

## 7. Pull ACS education data and merge

In [ ]:
acs_education_url = 'https://api.census.gov/data/2024/acs/acs5/subject?get=NAME,S1501_C02_015E&for=county:*'

acs_education = pd.read_json(acs_education_url)
acs_education.columns = acs_education.iloc[0]
acs_education = acs_education[1:].copy()

acs_education = acs_education.rename(columns={
    'NAME': 'county_name',
    'S1501_C02_015E': 'bachelors_or_higher_pct',
    'state': 'state_fips',
    'county': 'county_fips'
})

acs_education['state_fips'] = acs_education['state_fips'].astype(str).str.zfill(2)
acs_education['county_fips'] = acs_education['county_fips'].astype(str).str.zfill(3)
acs_education['fips'] = acs_education['state_fips'] + acs_education['county_fips']
acs_education['bachelors_or_higher_pct'] = pd.to_numeric(acs_education['bachelors_or_higher_pct'], errors='coerce')

acs_education = acs_education[['fips', 'county_name', 'bachelors_or_higher_pct']]
acs_education.head()

In [ ]:
merged = merged.drop(columns=['bachelors_or_higher_pct'], errors='ignore')

merged = merged.merge(
    acs_education[['fips', 'bachelors_or_higher_pct']],
    on='fips',
    how='left'
)

print('Missing bachelors_or_higher_pct values:', merged['bachelors_or_higher_pct'].isna().sum())
merged[merged['bachelors_or_higher_pct'].isna()][['state', 'county']].drop_duplicates()

## 8. Final cleaned dataset

In [ ]:
final_df = merged.dropna(subset=[
    'violent_crime',
    'unemployment_rate',
    'median_household_income',
    'poverty_rate',
    'bachelors_or_higher_pct'
]).copy()

print('Merged shape:', merged.shape)
print('Final shape after dropping rows with missing modeling variables:', final_df.shape)

final_df.head()

In [ ]:
final_df.to_csv(CLEAN_DATA / 'violent_crime_final_dataset.csv', index=False)
merged.to_csv(CLEAN_DATA / 'crime_bls_income_poverty_education_merged.csv', index=False)

## 9. Recommended next step

You are ready to move into exploratory analysis and modeling. A good next section would include:
- summary statistics
- histograms for each predictor
- a correlation matrix
- scatterplots of violent crime versus each socioeconomic variable
- a regression model to see which variables are the strongest predictors
